# Evaluation on labelled photos

How well each check and each rule does on photos with known labels, and why the thresholds are where they are. `pipeline_overview.ipynb` shows the pipeline on single photos.

**The photos.** The [Face Research Lab London Set](https://doi.org/10.6084/m9.figshare.5047666) (DeBruine & Jones 2017, CC BY 4.0): 102 people, photographed frontally on a plain grey background, neutral and smiling, plus 3/4 views. The subjects consented to use "in lab-based and web-based studies", so the photos stay outside this repository and this notebook shows no faces. `scripts/build_eval_set.py` makes from every person:

- **should pass**: the neutral photo cropped to 35x45 mm with the head at 75% of the height (`compliant`), the same tilted by 4° (`roll_4`), and with drawn-on clear glasses (`clear_glasses`);
- **real defects**: the smiling photo (`smiling`) and the two 3/4 views (`yaw_left`, `yaw_right`), cropped the same way;
- **synthetic defects**, each breaking one rule: tilted 12°, darkened, brightened, low contrast, blurred (σ 2 and 4 px), JPEG quality 4, head too small, off centre, red eyes, sunglasses, reflections on glasses, patterned background, objects in the background, colour cast, shadow on one side of the face, background darkening from left to right (`bg_gradient`), a shadow cast next to the person (`bg_shadow`), ink spots and a scratch on the background (`ink_stain`), skin flattened by an edge-preserving filter with the eyes spared (`smoothed`) or over the whole face (`smoothed_all`), coloured stickers on the nose and cheeks as a face filter paints them (`face_stickers`).

Odd person numbers are the `calibrate` split the thresholds were chosen on, even ones the `test` split.

**Labels from a vision model.** Claude labelled every neutral photo for what a passport clerk would reject (glasses, hair over the eyes, gaze, smile, head pose, shadows, ...), as a teacher for the subtle cases the folder names don't label. See section 5.

To reproduce (OFIQ built as described in `src/passphoto_validation/core/ofiq.py`):

```bash
uv run python scripts/build_eval_set.py --source <frl-london> --out <frl-london-eval>
OFIQ_CONFIG_DIR=<OFIQ>/data uv run python scripts/run_eval.py --set <frl-london-eval>
```

`results_classic.jsonl` is the same run of the pipeline before OFIQ (commit c92eac6), for comparison.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
import eval_report as er

EVAL = Path(os.environ.get('EVAL_SET', Path.cwd().parents[1] / 'datasets' / 'frl-london-eval'))
LONDON = EVAL.parent / 'frl-london'
OFIQ_DATA = Path(os.environ.get('OFIQ_CONFIG_DIR', Path.cwd().parents[1] / 'OFIQ-Project' / 'data'))

new_photos = er.load_photos(EVAL / 'results.jsonl')
old_photos = er.load_photos(EVAL / 'results_classic.jsonl')
new_rules, old_rules = er.rules_frame(new_photos), er.rules_frame(old_photos)
new_checks, old_checks = er.checks_frame(new_photos), er.checks_frame(old_photos)
people = pd.read_csv(LONDON / 'london_faces_info.csv', dtype={'face_id': str}).set_index('face_id')
labels = pd.read_json(EVAL / 'vlm_labels.jsonl', lines=True)
labels['identity'] = labels['file'].str[:3]
labels = labels.set_index('identity')

COMPLIANT, DEFECT, MUTED = '#2a78d6', '#eb6834', '#52514e'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True})
percent = '{:.0%}'.format
print(f'{len(new_photos)} photos of {len(people)} people; {len(labels)} labelled')

## 1. Old pipeline vs new, per rule

For every defect: how often the rule it breaks ends as **fail** (`caught`) or only **warn** (`warned`), and how often photos that should pass get a fail or warn on that rule (`false_fail`, `false_warn`). `_old` is the pipeline before OFIQ; empty where the old pipeline had no check for the rule.

In [ ]:
new = er.rule_summary(new_rules)
old = er.rule_summary(old_rules)
cols = ['caught', 'warned', 'false_fail', 'false_warn']
table = new[cols].join(old[cols], rsuffix='_old')
for c in cols:
    # The old pipeline reported a rule without a covering check as not_checked: show that as empty
    table.loc[old['not_judged'].reindex(table.index).fillna(1).eq(1), c + '_old'] = np.nan
table.style.format(percent, na_rep='').background_gradient(subset=['caught', 'caught_old'], cmap='Blues', vmin=0, vmax=1)

## 2. False alarms on photos that should pass

The share of `compliant`, `roll_4` and `clear_glasses` photos that pass every rule, and which rules raise the false alarms. A rule that fails good photos costs a retake every time, so this matters as much as catching defects.

In [ ]:
def clean_share(rules, statuses):
    good = rules[rules['variant'].isin(er.SHOULD_PASS)]
    bad_photo = good.groupby('file')['status'].agg(lambda s: s.isin(statuses).any())
    return 1 - bad_photo.mean()

print(f"No rule fails:           old {percent(clean_share(old_rules, ['fail']))}, new {percent(clean_share(new_rules, ['fail']))}")
print(f"No rule fails or warns:  old {percent(clean_share(old_rules, ['fail', 'warn']))}, new {percent(clean_share(new_rules, ['fail', 'warn']))}")
alarms = er.false_alarms(new_rules).join(er.false_alarms(old_rules), rsuffix='_old')
alarms[(alarms > 0).any(axis=1)].style.format(percent, na_rep='')

## 3. The checks and their thresholds

Each panel shows a check's measure on the photos that should pass (blue) and on the defect it is aimed at (orange), with the warn and fail thresholds as dashed lines. OFIQ quality scores run from 0 to 100, higher is better; pose is in degrees.

In [ ]:
from passphoto_validation.core.thresholds import thresholds, warn_fail

# (check, measurement, defect variant, warn threshold, fail threshold, label); the thresholds are the ones the
# validator runs with (core/thresholds.py), so a change there moves the lines here
PANELS = [
    ('ofiq_expression', 'ofiq_expression_neutrality_quality', 'smiling', *warn_fail('ofiq_expression'), 'Expression neutrality (OFIQ)'),
    ('ofiq_head_yaw', 'head_yaw_degrees_ofiq', 'yaw_left', *warn_fail('ofiq_head_yaw'), 'Head yaw, degrees (OFIQ)'),
    ('ofiq_head_roll', 'head_roll_degrees_ofiq', 'roll_12', *warn_fail('ofiq_head_roll'), 'Head roll, degrees (OFIQ)'),
    ('ofiq_illumination', 'ofiq_illumination_uniformity_quality', 'side_shadow', *warn_fail('ofiq_illumination'), 'Illumination uniformity (OFIQ)'),
    ('ofiq_background', 'ofiq_background_uniformity_quality', 'bg_pattern', *warn_fail('ofiq_background'), 'Background uniformity (OFIQ)'),
    ('ofiq_natural_colour', 'ofiq_natural_colour_quality', 'colour_cast', *warn_fail('ofiq_natural_colour'), 'Natural colour (OFIQ)'),
    ('colour_cast', 'colour_cast', 'colour_cast', *warn_fail('colour_cast'), 'Chroma the background and eye whites share'),
    ('exposure', 'highlight_lightness', 'dark', *warn_fail('exposure'), 'Lightness of the brightest 1%'),
    ('face_sharpness', 'face_sharpness', 'blur_2', *warn_fail('face_sharpness'), 'Face sharpness'),
    ('compression', 'jpeg_blockiness', 'jpeg', *warn_fail('compression'), 'JPEG blockiness'),
    ('glasses_tint', 'lens_brightness_ratio', 'sunglasses', *warn_fail('glasses_tint'), 'Under-eye / cheek lightness'),
    ('red_eyes', 'red_eye_share', 'red_eye', *warn_fail('red_eyes'), 'Red share of the pupil'),
    ('gaze', 'gaze_horizontal', None, *warn_fail('gaze'), 'Horizontal gaze offset'),
    ('background_gradient', 'background_gradient', 'bg_gradient', *warn_fail('background_gradient'), 'Background gradient, L* across the photo'),
    ('background_shadow', 'background_shadow_depth', 'bg_shadow', *warn_fail('background_shadow'), 'Shadow depth next to the person, L*'),
    ('background_shadow', 'background_shadow_rim_depth', 'bg_shadow', *warn_fail('background_shadow'), 'Shadow depth in the rim of the person mask, L*'),
    # Any mark warns (not a threshold of the check), three fail
    ('print_defects', 'background_marks', 'ink_stain', *warn_fail('print_defects'), 'Marks on the background'),
    ('skin_smoothing', 'skin_texture_ratio', 'smoothed', *warn_fail('skin_smoothing'), 'Skin / eye texture ratio (warns below)'),
    ('skin_smoothing', 'skin_gradient_ratio', 'smoothed_all', thresholds('skin_smoothing')['warn_gradient_ratio_below'], None, 'Skin texture / face gradient (warns below)'),
    ('skin_colour_patches', 'skin_foreign_share', 'face_stickers', *warn_fail('skin_colour_patches'), 'Share of the face in non-skin colours (warns above)'),
]

fig, axes = plt.subplots(5, 4, figsize=(16, 15.5), constrained_layout=True)
for ax in axes.flat[len(PANELS):]:
    ax.set_visible(False)
for ax, (name, column, defect, warn, fail, title) in zip(axes.flat, PANELS):
    data = new_checks[new_checks['check'] == name]
    groups = [('should pass', data[data['variant'].isin(er.SHOULD_PASS)], COMPLIANT)]
    if defect:
        groups.append((defect, data[data['variant'] == defect], DEFECT))
    values = [g[column].dropna().astype(float) for _, g, _ in groups if column in g]
    if not values or all(v.empty for v in values):
        ax.set_title(f'{title}\n(no values)'); continue
    if name in ('ofiq_head_yaw', 'ofiq_head_roll', 'gaze'):
        values = [v.abs() for v in values]
    bins = np.linspace(min(v.min() for v in values), max(v.max() for v in values), 40)
    for (label, _, color), v in zip(groups, values):
        ax.hist(v, bins=bins, color=color, alpha=0.75, label=f'{label} (n={len(v)})', edgecolor='white', linewidth=0.5)
    for threshold, kind in ((warn, 'warn'), (fail, 'fail')):
        if threshold is not None:
            ax.axvline(threshold, color=MUTED, linestyle='--' if kind == 'warn' else '-', linewidth=1)
            ax.annotate(kind, (threshold, ax.get_ylim()[1] * 0.92), color=MUTED, fontsize=8, ha='left', xytext=(3, 0), textcoords='offset points')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8, frameon=False)
plt.show()


The thresholds were chosen on the `calibrate` split. The same numbers on the `test` split, per check, show whether they hold on people they weren't chosen on:

In [ ]:
split = {s: er.check_summary(new_checks[new_checks['split'] == s]).set_index(['defect', 'check']) for s in ('calibrate', 'test')}
split['calibrate'][['false_fail', 'false_warn', 'caught', 'warned']].join(
    split['test'][['false_fail', 'false_warn', 'caught', 'warned']], lsuffix=' (cal)', rsuffix=' (test)'
).style.format(percent)

## 4. Exposure and skin tone

OFIQ's `UnderExposurePrevention` counts the dark pixels of the face, so it can't tell a dark-skinned face in good light from an underexposed one. The old check (mean skin brightness) has the same problem. The `exposure` check uses the brightest 1% of the photo instead, which in an ID photo is the light background. This cell runs OFIQ once more on the compliant photos, so it needs OFIQ.

In [ ]:
import cv2
from passphoto_validation.core.ofiq import load_engine

engine, why = load_engine(str(OFIQ_DATA))
rows = []
for identity in sorted(people.index):
    path = EVAL / f'{identity}_compliant.jpg'
    if not path.exists():
        continue
    row = {'identity': identity, 'group': 'Black' if people.loc[identity, 'face_eth'] == 'black' else 'all others'}
    if engine:
        row['OFIQ UnderExposure quality'] = engine.assess(cv2.imread(str(path))).measures['UnderExposurePrevention'].quality
    rows.append(row)
skin = pd.DataFrame(rows).set_index('identity')
def compliant_measure(checks, name, column):
    c = checks[(checks['variant'] == 'compliant') & (checks['check'] == name)]
    return c.set_index('identity')[column] if column in c else None
skin['old: mean skin brightness'] = compliant_measure(old_checks, 'underexposure', 'face_brightness')
skin['new: highlight lightness'] = compliant_measure(new_checks, 'exposure', 'highlight_lightness')
print('' if engine else f'OFIQ not loaded: {why}')
skin.groupby('group').agg(['count', 'min', 'median']).round(1)

## 5. What the vision-model labels add

The folder names say *smiling* or *3/4 view*; they don't say whether a neutral photo would really pass. The labels do, and they catch borderline cases: faint closed-mouth smiles, a raised chin, a fringe over the eyes. Below: the photos labelled not compliant, and what the pipeline says about them (the `compliant` crop of the same photo).

In [ ]:
compliant = new_checks[new_checks['variant'] == 'compliant']
def column(name, measurement):
    c = compliant[compliant['check'] == name].set_index('identity')
    return c[measurement] if measurement in c else c['status']
view = pd.DataFrame({
    'expression': column('ofiq_expression', 'ofiq_expression_neutrality_quality'),
    'mouth closed': column('ofiq_mouth_closed', 'ofiq_mouth_closed_quality'),
    'pitch °': column('ofiq_head_pitch', 'head_pitch_degrees_ofiq').round(1),
    'eyes visible': column('ofiq_eyes_visible', 'ofiq_eyes_visible_quality'),
    'rules failed': new_rules[(new_rules['variant'] == 'compliant') & (new_rules['status'] == 'fail')].groupby('identity')['rule'].agg(', '.join),
    'rules warned': new_rules[(new_rules['variant'] == 'compliant') & (new_rules['status'] == 'warn')].groupby('identity')['rule'].agg(', '.join),
}).join(labels[['compliant', 'note']])
pd.set_option('display.max_colwidth', 120)
view[~view['compliant']].fillna('')

Agreement between a label and the check aimed at it, on the 102 compliant crops. `label says problem` counts photos the labeller flagged; `check flags` counts how many of those the check warned or failed on, and `check flags, label doesn't` the reverse.

In [ ]:
PAIRS = [
    ('expression_neutral', False, 'ofiq_expression'),
    ('mouth_closed', False, 'ofiq_mouth_closed'),
    ('head_turned', True, 'ofiq_head_pitch'),
    ('head_tilted', True, 'ofiq_head_roll'),
    ('hair_over_eyes', True, 'ofiq_eyes_visible'),
    ('gaze_at_camera', False, 'gaze'),
]
rows = []
for field, problem_value, name in PAIRS:
    status = compliant[compliant['check'] == name].set_index('identity')['status']
    flagged = status.isin(['warn', 'fail'])
    problem = labels[field] == problem_value
    both = pd.concat([flagged.rename('check'), problem.rename('label')], axis=1).dropna()
    rows.append({'label': field, 'check': name, 'label says problem': int(both['label'].sum()),
                 'check flags': int((both['label'] & both['check']).sum()),
                 "check flags, label doesn't": int((~both['label'] & both['check']).sum())})
pd.DataFrame(rows).set_index(['label', 'check'])

## 6. What this doesn't show

- **One studio.** Every photo has the same light, background and camera. Real submissions (phones, kiosks, scans) will spread the measures wider, so expect more false alarms than here.
- **Synthetic defects are clean.** A drawn-on pair of sunglasses or a pasted pattern is easier than the real thing. Glasses in particular: nobody in the set wears any, so clear glasses and reflections are tested only on drawn-on frames, plus the one real pair in `tests/test-images/9.jpg`.
- **Not covered by any variant:** hats and head coverings, averted gaze, closed eyes, shadows from above. Hats, real glasses, averted gaze and closed eyes are tested on 240 Open Images photos instead (`scripts/score_labels.py`, results in `PASSPORT_VALIDATOR.md`); the rest run with OFIQ's or plausible thresholds, untested.
- **The youngest checks** (`background_gradient`, `background_shadow`, `print_defects`, `skin_smoothing`, `skin_colour_patches`) are tested on synthetic defects only: a linear darkening, a shadow drawn next to the silhouette, drawn ink spots, an edge-preserving filter on the skin or the whole face, drawn stickers on the nose and cheeks. Real backdrops, prints, beauty filters and face filters will differ, and the shadow margin is thin (compliant crops reach a depth of 9 against a fail threshold of 9). The sticker check sees only saturated colours on the lower face: white sparkles, black whiskers, ears and crowns above the head pass it.
- **The labels are one model's judgement.** Two labelling batches disagreed on what a "slight" face shadow is, so that field isn't used. The faint-smile calls are borderline by the labeller's own account.